## Imports

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('kaggle', 'timm', 'tifffile', 'torchstain', 'scikit-learn')

# ==============================================================================
# IMPORTS
# ==============================================================================
import os, gc, glob, json, pathlib, zipfile, urllib.request, time, warnings, random
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.backends import cudnn
import timm
from timm.layers import SwiGLUPacked

cudnn.benchmark = True
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## Config

In [ ]:
KAGGLE_TOKEN    = 'place holder'
KAGGLE_USERNAME = 'place holder'
# IMPORTANT: Must be a classic Read token (not fine-grained) so it can access
# gated repos like paige-ai/Virchow. Create one at:
#   https://huggingface.co/settings/tokens → New token → Read (classic)
# Also visit https://huggingface.co/paige-ai/Virchow and accept the terms first.
HF_TOKEN        = 'place holder'  # ← replace with classic token

os.environ['HF_TOKEN'] = HF_TOKEN

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = pathlib.Path('./virchow_run')
DATA_DIR    = BASE_DIR / 'data'
EXT_DIR     = BASE_DIR / 'data' / 'external'
MODELS_DIR  = BASE_DIR / 'models'
for d in [DATA_DIR, EXT_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUBMISSION_FILE = 'submission_virchow.csv'

# ── Model / training ──────────────────────────────────────────────────────────
IMG_SIZE   = (784, 784)   # must be divisible by 14 (Virchow patch size)
EPOCHS     = 30
LR         = 2e-4         # same as hubmap_pipeline.py
FOLDS      = 5
TRAIN_FOLD = 0            # same fold as ablation / film_no_external for comparison
UNFREEZE_BLOCKS = 6       # last N Virchow transformer blocks to fine-tune (H100 headroom)

# Auto-scale batch/accumulation to available GPU VRAM
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    n_gpus  = torch.cuda.device_count()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 70:          # H100 80GB
        BATCH_SIZE, ACCUM = 8, 2
    elif vram_gb >= 35:        # A100 40GB
        BATCH_SIZE, ACCUM = 4, 4
    else:                      # smaller
        BATCH_SIZE, ACCUM = 2, 8
    print(f"GPUs={n_gpus}  VRAM={vram_gb:.0f}GB  "
          f"batch={BATCH_SIZE}  accum={ACCUM}  "
          f"effective_batch={BATCH_SIZE * n_gpus * ACCUM}")
else:
    n_gpus = 0; BATCH_SIZE = 2; ACCUM = 8

ORGAN_MAP  = {'kidney': 0, 'prostate': 1, 'largeintestine': 2, 'spleen': 3, 'lung': 4}
SOURCE_MAP = {'Hubmap': 0, 'HPA': 1}

ORGAN_PIXEL_SIZE_DEFAULTS = {
    'kidney': 0.50, 'prostate': 0.27, 'largeintestine': 0.23,
    'spleen': 0.49, 'lung': 0.50,
}
REF_PIXEL_SIZE = 0.4

ORGAN_THRESHOLDS = {
    'Hubmap': {'kidney': 90,  'prostate': 100, 'largeintestine': 80, 'spleen': 100, 'lung': 15},
    'HPA':    {'kidney': 127, 'prostate': 127, 'largeintestine': 127,'spleen': 127, 'lung': 25},
}


## Kaggle Auth

In [ ]:
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

_kdir = pathlib.Path.home() / '.kaggle'
_kdir.mkdir(parents=True, exist_ok=True)
_token_path = _kdir / 'access_token'
_token_path.write_text(KAGGLE_TOKEN)
os.chmod(_token_path, 0o600)
print(f"✓ Kaggle auth configured (user: {KAGGLE_USERNAME})")


def _kaggle_dl(cmd, max_retries=6):
    wait = 60
    for attempt in range(max_retries):
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            return r
        combined = (r.stdout + r.stderr).strip()
        if '429' in combined or 'Too Many Requests' in combined:
            if attempt < max_retries - 1:
                print(f"  Rate limited — waiting {wait}s… ({attempt+1}/{max_retries-1})")
                time.sleep(wait); wait = min(wait*2, 300)
                continue
        raise subprocess.CalledProcessError(
            r.returncode, cmd, output=r.stdout, stderr=r.stderr)
    raise subprocess.CalledProcessError(max_retries, cmd, output='', stderr='Max retries exceeded')


## Data

In [ ]:
_train_csv = DATA_DIR / 'train.csv'
if not _train_csv.exists():
    print("Downloading HuBMAP competition data…")
    try:
        _kaggle_dl(['kaggle', 'competitions', 'download',
                    '-c', 'hubmap-organ-segmentation', '-p', str(DATA_DIR)])
    except subprocess.CalledProcessError as e:
        _msg = ((e.stderr or '') + (e.stdout or '')).strip() or '(no output)'
        raise RuntimeError(
            f"Competition download failed.\n"
            f"  Kaggle says: {_msg[:500]}\n\n"
            f"  Common causes:\n"
            f"    • Token expired → update KAGGLE_TOKEN at top of script\n"
            f"    • Rules not accepted → visit kaggle.com/c/hubmap-organ-segmentation/rules\n"
            f"    • Not joined the competition → click 'Join Competition' on Kaggle\n"
        )
    for z in glob.glob(str(DATA_DIR / '*.zip')):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA_DIR)
        os.remove(z)
    print("✓ Competition data ready")
else:
    print("Competition data cached.")

HUBMAP_IMG_DIR = DATA_DIR / 'train_images'

# ── Zenodo Team_2 lung data
ZENODO_URL      = 'https://zenodo.org/records/7545745/files/Team_2.zip'
LUNG_DIR        = EXT_DIR / 'zenodo_lung'
_lung_done      = LUNG_DIR / '.done'
_lung_manifest  = LUNG_DIR / 'manifest.json'

if not _lung_done.exists():
    LUNG_DIR.mkdir(parents=True, exist_ok=True)
    print(f"\nDownloading Zenodo Team_2 lung data ({ZENODO_URL})…")
    _zip = LUNG_DIR / 'Team_2.zip'
    req = urllib.request.urlopen(ZENODO_URL, timeout=120)
    total = int(req.headers.get('Content-Length', 0))
    downloaded = 0
    with open(_zip, 'wb') as f:
        while chunk := req.read(1024*1024):
            f.write(chunk); downloaded += len(chunk)
            if total:
                print(f"\r  {downloaded/1e6:.0f}/{total/1e6:.0f} MB", end='', flush=True)
    print()
    print("  Extracting lung files…")
    n_ext = 0
    with zipfile.ZipFile(_zip) as zf:
        for m in zf.namelist():
            if 'lung' in m.lower() and not m.endswith('/'):
                zf.extract(m, LUNG_DIR); n_ext += 1
    _zip.unlink()

    # Build manifest: find image files, try to get RLE from pseudo-label CSVs
    # (CSVs live in the vladimirsydor pseudo-label dataset — if not available,
    #  we use mask PNG files that Team_2 provides alongside images)
    IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
    manifest = []
    for root, _, fnames in os.walk(LUNG_DIR):
        for fn in fnames:
            if 'mask' in fn.lower(): continue
            if os.path.splitext(fn)[1].lower() not in IMAGE_EXTS: continue
            img_p  = os.path.join(root, fn)
            stem   = os.path.splitext(fn)[0]
            mask_p = os.path.join(root, stem + '_mask.png')
            manifest.append({'img_path': img_p,
                             'mask_path': mask_p if os.path.exists(mask_p) else None,
                             'organ': 'lung', 'data_source': 'HPA',
                             'pixel_size': 0.50})
    _lung_manifest.write_text(json.dumps(manifest, indent=2))
    _lung_done.touch()
    print(f"✓ Zenodo lung: {len(manifest)} images ready  "
          f"({sum(1 for r in manifest if r['mask_path']) } with mask PNGs)")
else:
    manifest = json.loads(_lung_manifest.read_text())
    print(f"Zenodo lung cached: {len(manifest)} images")

# ── Patch Zenodo manifest with RLEs from hpa_lungs.csv ───────────────────────
# The Zenodo images have no _mask.png files. Their masks live as RLE strings in
# hpa_lungs.csv (part of the vladimirsydor dataset). We download ONLY that one
# CSV file here (not the full dataset) so we can match image stems → RLEs.
# We do NOT add any HPA lung images to training (test set is HuBMAP-only).
_lung_rle_lookup: dict = {}
_lung_csv_dl = EXT_DIR / 'hpa_lungs.csv'
if not _lung_csv_dl.exists():
    print("  Downloading hpa_lungs.csv for Zenodo RLE lookup…")
    try:
        _kaggle_dl(['kaggle', 'datasets', 'download',
                    '-d', 'vladimirsydor/hubmap-2022-add-data-labels-v2',
                    '--file', 'hpa_lungs.csv',
                    '-p', str(EXT_DIR)])
        # kaggle wraps single-file downloads in a zip
        for _z in glob.glob(str(EXT_DIR / '*.zip')):
            with zipfile.ZipFile(_z) as _zf:
                _zf.extractall(EXT_DIR)
            os.remove(_z)
        print("  ✓ hpa_lungs.csv downloaded")
    except Exception as _e:
        print(f"  [WARN] Could not fetch hpa_lungs.csv ({_e}) — Zenodo RLEs unavailable")

if _lung_csv_dl.exists():
    try:
        _ldf = pd.read_csv(_lung_csv_dl)
        if 'encoding' in _ldf.columns and 'rle' not in _ldf.columns:
            _ldf = _ldf.rename(columns={'encoding': 'rle'})
        if 'id' in _ldf.columns and 'rle' in _ldf.columns:
            for _, _r in _ldf.iterrows():
                if pd.notna(_r.get('rle')) and str(_r['rle']) not in ('', 'nan'):
                    _lung_rle_lookup[str(_r['id'])] = str(_r['rle'])
        print(f"  Lung RLE lookup: {len(_lung_rle_lookup)} entries from hpa_lungs.csv")
    except Exception as _e:
        print(f"  [WARN] Could not parse hpa_lungs.csv: {_e}")

# Patch manifest entries that have no mask_path with RLEs from the lookup
_patched = 0
for _entry in manifest:
    if _entry.get('mask_path') or _entry.get('rle'):
        continue
    _stem = os.path.splitext(os.path.basename(_entry['img_path']))[0]
    _rle  = _lung_rle_lookup.get(_stem)
    if _rle is None:
        # Try suffix match (some IDs have prefix differences)
        _sfx = _stem.split('_')[-1]
        for _k, _v in _lung_rle_lookup.items():
            if _k.endswith('_' + _sfx) or _k == _sfx:
                _rle = _v; break
    if _rle:
        _entry['rle'] = _rle; _patched += 1

if _patched:
    _lung_manifest.write_text(json.dumps(manifest, indent=2))
    print(f"  Patched {_patched} Zenodo entries with RLEs from hpa_lungs.csv")

LUNG_MANIFEST = [r for r in manifest if r['mask_path'] or r.get('rle')]
print(f"  Usable lung entries (with supervision): {len(LUNG_MANIFEST)}")

# ── HPA pseudo-labels (kidney / prostate / large intestine only) ──────────────
# Dataset: vladimirsydor/hubmap-2022-add-data-labels-v2
# We SKIP hpa_lungs.csv and hpa_spleen.csv (too far from HuBMAP test distribution).
# Download is optional — if the dataset is inaccessible (403/private) we continue
# with HuBMAP + Zenodo lung only. To fix a 403, visit:
#   https://www.kaggle.com/datasets/vladimirsydor/hubmap-2022-add-data-labels-v2
# and click Download while logged in as erickgonz.
_SKIP_CSVS   = {'hpa_lungs.csv', 'hpa_spleen.csv'}
PSEUDO_DIR   = EXT_DIR / 'pseudo_labels'
_pseudo_done = PSEUDO_DIR / '.done'
_pseudo_ok   = False  # set to True if download succeeds or data is cached

if not _pseudo_done.exists():
    PSEUDO_DIR.mkdir(parents=True, exist_ok=True)
    print("\nDownloading HPA pseudo-label dataset…")
    try:
        _kaggle_dl(['kaggle', 'datasets', 'download',
                    '-d', 'vladimirsydor/hubmap-2022-add-data-labels-v2',
                    '-p', str(PSEUDO_DIR)])
        for z in glob.glob(str(PSEUDO_DIR / '*.zip')):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(PSEUDO_DIR)
            os.remove(z)
        _pseudo_done.touch()
        _pseudo_ok = True
        print("✓ HPA pseudo-labels ready")
    except subprocess.CalledProcessError as e:
        _msg = ((e.stderr or '') + (e.stdout or '')).strip()
        print(f"  [WARN] HPA pseudo-label download failed (will train without them).")
        print(f"  Kaggle says: {_msg[:300]}")
        print(f"  Fix: visit kaggle.com/datasets/vladimirsydor/hubmap-2022-add-data-labels-v2"
              f" and click Download while logged in as {KAGGLE_USERNAME}.")
else:
    _pseudo_ok = True
    print("HPA pseudo-labels cached.")

# Index CSVs and build HPA_ROWS only if download succeeded
HPA_ROWS = []
if _pseudo_ok and PSEUDO_DIR.exists():
    _pseudo_img_roots = [d for d in PSEUDO_DIR.iterdir()
                         if d.is_dir() and 'images' in d.name.lower()]
    if not _pseudo_img_roots:
        _pseudo_img_roots = [PSEUDO_DIR]

    for csv_p in sorted(PSEUDO_DIR.glob('*.csv')):
        if csv_p.name in _SKIP_CSVS:
            print(f"  [SKIP] {csv_p.name}")
            continue
        df_p = pd.read_csv(csv_p)
        for _, row in df_p.iterrows():
            img_file = None
            for root_dir in _pseudo_img_roots:
                candidates = list(root_dir.glob(f"{row['id']}.*"))
                if candidates:
                    img_file = str(candidates[0]); break
            if img_file is None:
                for ext in ('.tiff', '.tif', '.png', '.jpg'):
                    p = PSEUDO_DIR / f"{row['id']}{ext}"
                    if p.exists():
                        img_file = str(p); break
            if img_file is None:
                continue
            organ = str(row.get('organ', row.get('organ_type', 'unknown'))).lower().replace(' ', '')
            HPA_ROWS.append({
                'id':          row['id'],
                'img_path':    img_file,
                'rle':         row.get('annotation', row.get('rle', '')),
                'organ':       organ,
                'data_source': 'HPA',
                'pixel_size':  ORGAN_PIXEL_SIZE_DEFAULTS.get(organ, 0.4),
                'img_height':  row.get('img_height', 0),
                'img_width':   row.get('img_width', 0),
            })

print(f"  HPA pseudo-label rows loaded (excluding lung/spleen): {len(HPA_ROWS)}")


## Utilities

In [ ]:
def rle_decode(rle, shape):
    s = rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[::2], s[1::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape[1], shape[0]).T


def rle_encode(img):
    pixels = img.T.flatten()
    pixels[0] = 0; pixels[-1] = 0
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 2
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


def _load_image_safe(path, max_dim=1024):
    """Pyramid-level TIFF loading — same as hubmap_pipeline.py."""
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.tiff', '.tif'):
        try:
            import tifffile
            with tifffile.TiffFile(path) as tf:
                series = tf.series[0]
                chosen = 0
                for lvl, level in enumerate(series.levels):
                    sh = level.shape
                    h = sh[-2] if len(sh) >= 2 else sh[0]
                    w = sh[-1]
                    if max(h, w) >= max_dim:
                        chosen = lvl
                    else:
                        break
                arr = series.levels[chosen].asarray()
                if arr.ndim == 3 and arr.shape[0] <= 4:
                    arr = np.transpose(arr, (1, 2, 0))
                return arr
        except Exception:
            pass
    return cv2.imread(path, cv2.IMREAD_UNCHANGED)


def ensure_3ch(img):
    if img is None: return np.zeros((*IMG_SIZE, 3), dtype=np.uint8)
    if img.dtype != np.uint8:
        lo, hi = img.min(), img.max()
        img = ((img - lo) / (hi - lo + 1e-8) * 255).clip(0,255).astype(np.uint8)
    if img.ndim == 2:    return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 1: return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 4: return cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    return img[:, :, :3]


def preprocess_inputs(x):
    """Scale to [-1, 1] — identical to hubmap_pipeline.py."""
    x = np.asarray(x, dtype='float32')
    x /= 127.0; x -= 1.0
    return x


# ── Stain normalization — identical to hubmap_pipeline.py ────────────────────
_STAIN_NORMALIZER = None

def _build_stain_normalizer(hubmap_img_dir):
    global _STAIN_NORMALIZER
    if _STAIN_NORMALIZER is not None: return
    imgs = sorted(glob.glob(str(pathlib.Path(hubmap_img_dir) / '*.tiff')))
    if not imgs: return
    ref_bgr = cv2.imread(imgs[0], cv2.IMREAD_COLOR)
    if ref_bgr is None: return
    ref_rgb = cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2RGB)
    ref_t   = torch.from_numpy(ref_rgb).permute(2, 0, 1).float()
    try:
        import torchstain
        norm = torchstain.normalizers.MacenkoNormalizer(backend='torch')
        norm.fit(ref_t)
        _STAIN_NORMALIZER = norm
        print(f"  Stain normalizer fitted on {os.path.basename(imgs[0])}")
    except Exception as e:
        print(f"  [WARN] Macenko fit failed ({e}) — disabled.")


def _normalize_stain(img_bgr):
    if _STAIN_NORMALIZER is None: return img_bgr
    try:
        rgb    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        t      = torch.from_numpy(rgb).permute(2, 0, 1).float()
        result = _STAIN_NORMALIZER.normalize(I=t, stains=True)
        normed = result[0] if isinstance(result, (tuple, list)) else result
        if not isinstance(normed, torch.Tensor) or normed.ndim != 3: return img_bgr
        np_img = normed.permute(1, 2, 0).clamp(0, 255).byte().numpy()
        return cv2.cvtColor(np_img, cv2.COLOR_RGB2BGR) if np_img.shape == img_bgr.shape else img_bgr
    except Exception:
        return img_bgr


# ── Augmentation — identical to hubmap_pipeline.py ───────────────────────────
_HEAVY_AUG_ORGANS = {'lung', 'spleen'}

def _augment(img, mask, organ):
    if np.random.rand() > 0.5:
        img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
    if np.random.rand() > 0.5:
        img, mask = np.flipud(img).copy(), np.flipud(mask).copy()
    if organ in _HEAVY_AUG_ORGANS:
        k = np.random.choice([0,1,2,3])
        if k:
            img  = np.rot90(img,  k=k).copy()
            mask = np.rot90(mask, k=k).copy()
        alpha = 1.0 + np.random.uniform(-0.15, 0.15)
        beta  = np.random.uniform(-20, 20)
        img   = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        if np.random.rand() > 0.6:
            ksize = np.random.choice([3, 5])
            img   = cv2.GaussianBlur(img, (ksize, ksize), 0)
    return img, mask


## Manifest

In [ ]:
def build_manifest():
    # HuBMAP competition rows
    df = pd.read_csv(DATA_DIR / 'train.csv')
    rows = []
    for _, r in df.iterrows():
        p = HUBMAP_IMG_DIR / f"{r['id']}.tiff"
        if not p.exists() or pd.isna(r.get('rle', float('nan'))): continue
        rows.append({'img_path': str(p), 'rle': r['rle'], 'organ': r['organ'],
                     'data_source': 'Hubmap',
                     'pixel_size': ORGAN_PIXEL_SIZE_DEFAULTS.get(r['organ'], 0.40)})

    n_hub = len(rows)

    # HPA pseudo-label rows (kidney / prostate / largeintestine only)
    for entry in HPA_ROWS:
        rows.append({
            'img_path':    entry['img_path'],
            'rle':         entry.get('rle', ''),
            'organ':       entry['organ'],
            'data_source': 'HPA',
            'pixel_size':  entry.get('pixel_size', 0.4),
            'img_height':  entry.get('img_height', 0),
            'img_width':   entry.get('img_width', 0),
        })

    n_hpa = len(rows) - n_hub

    # Zenodo lung rows (mask PNG supervision)
    for entry in LUNG_MANIFEST:
        rows.append({'img_path': entry['img_path'],
                     'mask_path': entry.get('mask_path'),
                     'rle': entry.get('rle'),
                     'organ': 'lung', 'data_source': 'HPA',
                     'pixel_size': 0.50})

    n_lng = len(rows) - n_hub - n_hpa
    print(f"Manifest: {len(rows)} rows  "
          f"({n_hub} HuBMAP  {n_hpa} HPA pseudo  {n_lng} Zenodo lung)")
    return pd.DataFrame(rows)


print("\nBuilding manifest…")
MANIFEST = build_manifest()
_build_stain_normalizer(HUBMAP_IMG_DIR)

## Dataset

In [ ]:
class HubMapDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df       = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load image
        img = _load_image_safe(row['img_path'], max_dim=max(IMG_SIZE)*2)
        img = ensure_3ch(img)

        # Load mask
        rle = str(row.get('rle', ''))
        if rle and rle not in ('nan', ''):
            mask = rle_decode(rle, img.shape[:2])
        elif row.get('mask_path') and os.path.exists(str(row['mask_path'])):
            raw  = cv2.imread(str(row['mask_path']), cv2.IMREAD_UNCHANGED)
            if raw is None:
                mask = np.zeros(img.shape[:2], dtype=np.uint8)
            else:
                if raw.ndim != 2: raw = cv2.cvtColor(raw, cv2.COLOR_BGR2GRAY)
                mask = (raw > 15).astype(np.uint8)
        else:
            mask = np.zeros(img.shape[:2], dtype=np.uint8)

        # Pre-cap before pixel-scale resize
        MAX_PRE = max(IMG_SIZE) * 2
        if img.shape[0] > MAX_PRE or img.shape[1] > MAX_PRE:
            img  = cv2.resize(img,  (MAX_PRE, MAX_PRE))
            mask = cv2.resize(mask, (MAX_PRE, MAX_PRE), interpolation=cv2.INTER_NEAREST)

        # Pixel-scale normalization — identical to hubmap_pipeline.py
        pixel_size = float(row.get('pixel_size', REF_PIXEL_SIZE))
        if pixel_size > 0:
            scale = np.clip(pixel_size / REF_PIXEL_SIZE, 0.25, 4.0)
            if abs(scale - 1.0) > 0.05:
                new_h = max(64, int(img.shape[0] * scale))
                new_w = max(64, int(img.shape[1] * scale))
                img  = cv2.resize(img,  (new_w, new_h))
                mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

        # Final resize to training resolution
        img  = cv2.resize(img,  IMG_SIZE)
        mask = cv2.resize(mask, IMG_SIZE, interpolation=cv2.INTER_NEAREST)

        # Stain normalization — ALL images normalised toward HuBMAP reference.
        # Applying Macenko to HPA images brings their stain appearance closer to
        # the HuBMAP-only test set, reducing domain gap for all organ sources.
        img = _normalize_stain(img)

        if self.is_train:
            img, mask = _augment(img, mask, row['organ'])

        img_t  = torch.from_numpy(preprocess_inputs(img).transpose((2,0,1)).copy()).float()
        mask_t = torch.from_numpy(mask.copy()).float().unsqueeze(0)

        organ_idx   = torch.tensor(ORGAN_MAP.get(row['organ'], 0),  dtype=torch.long)
        source_idx  = torch.tensor(SOURCE_MAP.get(row.get('data_source','HPA'), 1),
                                   dtype=torch.long)
        pixel_scale = torch.tensor(pixel_size / REF_PIXEL_SIZE, dtype=torch.float32)

        return {'img': img_t, 'mask': mask_t, 'organ': row['organ'],
                'organ_idx': organ_idx, 'source_idx': source_idx,
                'pixel_scale': pixel_scale}


## Model

In [ ]:
class FiLM(nn.Module):
    def __init__(self, cond_dim, num_channels):
        super().__init__()
        self.fc = nn.Linear(cond_dim, num_channels * 2)

    def forward(self, x, condition):
        gb = self.fc(condition).unsqueeze(-1).unsqueeze(-1)
        g, b = torch.chunk(gb, 2, dim=1)
        return x * (1 + g) + b


class MetadataEmbedder(nn.Module):
    """Identical to hubmap_pipeline.py."""
    def __init__(self, cond_dim=128):
        super().__init__()
        self.organ_emb  = nn.Embedding(6, 32)
        self.source_emb = nn.Embedding(3, 16)
        self.scale_mlp  = nn.Sequential(nn.Linear(1, 16), nn.SiLU())
        self.fusion     = nn.Sequential(
            nn.Linear(64, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))

    def forward(self, organ_idx, source_idx, pixel_scale):
        return self.fusion(torch.cat([
            self.organ_emb(organ_idx),
            self.source_emb(source_idx),
            self.scale_mlp(pixel_scale.view(-1, 1)),
        ], dim=1))


class ConvSiluFiLM(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim, ks=3):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, ks, padding=1)
        self.bn   = nn.GroupNorm(min(32, out_ch), out_ch)  # GN safer than BN at small batch
        self.silu = nn.SiLU(inplace=True)
        self.film = FiLM(cond_dim, out_ch)

    def forward(self, x, c):
        return self.film(self.silu(self.bn(self.conv(x))), c)


class MultimodalVirchowUNet(nn.Module):
    """
    Virchow ViT-L/14 encoder + FiLM-conditioned decoder.

    Architecture differences vs MultimodalPathologyUNet (hubmap_pipeline.py):
      - Encoder: Virchow (ViT-L, 1280-dim) vs ConvNeXt-Base (hierarchical)
      - Decoder: 4-stage linear upsampling (no skip connections — ViT is flat)
      - GroupNorm instead of BatchNorm (critical for small batch sizes)
      - Auxiliary heads on CLS token rather than avg-pooled bottleneck

    Everything conditioning-related (MetadataEmbedder, FiLM, graceful
    degradation at inference) is identical to hubmap_pipeline.py.
    """
    def __init__(self, cond_dim=128, unfreeze_blocks=6):
        super().__init__()
        print("Loading paige-ai/Virchow from HuggingFace Hub…")
        self.encoder = timm.create_model(
            'hf-hub:paige-ai/Virchow',
            pretrained=True,
            mlp_layer=SwiGLUPacked,
            act_layer=nn.SiLU,
            dynamic_img_size=True,
        )
        self.embed_dim  = 1280
        self.patch_size = 14

        # Freeze all encoder params, then selectively unfreeze last N blocks
        for p in self.encoder.parameters():
            p.requires_grad = False
        n_blocks = len(self.encoder.blocks)
        for block in self.encoder.blocks[n_blocks - unfreeze_blocks:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.encoder.norm.parameters():
            p.requires_grad = True
        n_trainable = sum(p.numel() for p in self.encoder.parameters() if p.requires_grad)
        print(f"  Virchow: {n_blocks} blocks, last {unfreeze_blocks} unfrozen  "
              f"({n_trainable/1e6:.1f}M trainable encoder params)")

        self.metadata_embedder = MetadataEmbedder(cond_dim)

        # Auxiliary prediction heads (graceful degradation — same as hubmap_pipeline.py)
        self.pred_organ  = nn.Linear(self.embed_dim, 5)
        self.pred_source = nn.Linear(self.embed_dim, 2)
        self.pred_scale  = nn.Linear(self.embed_dim, 1)

        # 4-stage FiLM decoder: 1280 → 512 → 256 → 128 → 64 → 1
        # Each stage: upsample 2× then ConvSiluFiLM
        decoder_dims = [512, 256, 128, 64]
        self.dec1 = ConvSiluFiLM(self.embed_dim,     decoder_dims[0], cond_dim)
        self.dec2 = ConvSiluFiLM(decoder_dims[0],    decoder_dims[1], cond_dim)
        self.dec3 = ConvSiluFiLM(decoder_dims[1],    decoder_dims[2], cond_dim)
        self.dec4 = ConvSiluFiLM(decoder_dims[2],    decoder_dims[3], cond_dim)
        self.head = nn.Conv2d(decoder_dims[3], 1, 1)

        n_dec = sum(p.numel() for p in list(self.dec1.parameters()) +
                    list(self.dec2.parameters()) + list(self.dec3.parameters()) +
                    list(self.dec4.parameters()) + list(self.head.parameters()) +
                    list(self.metadata_embedder.parameters()))
        print(f"  Decoder + embedder: {n_dec/1e6:.1f}M params")

    def forward(self, x, gt_organ=None, gt_source=None, gt_scale=None):
        B, C, H, W = x.shape
        grid_h = H // self.patch_size  # 784 // 14 = 56
        grid_w = W // self.patch_size

        out = self.encoder.forward_features(x)
        cls_token      = out[:, 0, :]           # (B, 1280)
        spatial_tokens = out[:, 1:, :]          # (B, 56*56, 1280)

        # Auxiliary predictions from CLS token
        po  = self.pred_organ(cls_token)
        ps  = self.pred_source(cls_token)
        psc = self.pred_scale(cls_token)

        # Metadata embedding — use GT at train time, predicted at inference
        if gt_organ is not None:
            cond = self.metadata_embedder(gt_organ, gt_source, gt_scale)
        else:
            cond = self.metadata_embedder(torch.argmax(po, 1), torch.argmax(ps, 1), psc)

        # Reshape spatial tokens → 2D map: (B, 1280, 56, 56)
        feat = spatial_tokens.transpose(1, 2).contiguous().view(B, self.embed_dim, grid_h, grid_w)

        up = lambda t, sz: F.interpolate(t, size=sz, mode='bilinear', align_corners=False)
        # 56→112→224→392→784
        dec = self.dec1(up(feat, (112, 112)), cond)
        dec = self.dec2(up(dec,  (224, 224)), cond)
        dec = self.dec3(up(dec,  (392, 392)), cond)
        dec = self.dec4(up(dec,  (784, 784)), cond)
        return self.head(dec)


def build_model():
    m = MultimodalVirchowUNet(cond_dim=128, unfreeze_blocks=UNFREEZE_BLOCKS)
    return m

## Loss

In [ ]:
class FocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, dice_weight=0.5, smooth=1e-5):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
        self.dice_weight = dice_weight; self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets):
        bce  = self.bce(logits, targets)
        prob = torch.sigmoid(logits)
        pt   = prob * targets + (1 - prob) * (1 - targets)
        fw   = (1 - pt) ** self.gamma
        aw   = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        fl   = (aw * fw * bce).mean()
        i    = (prob * targets).sum(dim=(2, 3))
        u    = prob.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dl   = 1.0 - ((2.*i + self.smooth) / (u + self.smooth)).mean()
        return (1 - self.dice_weight) * fl + self.dice_weight * dl


def dice_score(pred, target, threshold=0.5):
    pred  = (torch.sigmoid(pred) > threshold).float()
    inter = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return ((2.*inter + 1e-5) / (union + 1e-5)).mean().item()


## Training

In [ ]:
def train():
    # ── Split — StratifiedKFold on HuBMAP only, append external data to train ──
    hubmap_df = MANIFEST[MANIFEST['data_source'] == 'Hubmap'].reset_index(drop=True)
    ext_df    = MANIFEST[MANIFEST['data_source'] != 'Hubmap'].reset_index(drop=True)
    skf       = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
    splits    = list(skf.split(hubmap_df, hubmap_df['organ']))
    tr_idx, va_idx = splits[TRAIN_FOLD]
    train_df  = pd.concat([hubmap_df.iloc[tr_idx], ext_df], ignore_index=True)
    val_df    = hubmap_df.iloc[va_idx].reset_index(drop=True)

    organ_counts = train_df['organ'].value_counts().to_dict()
    print(f"\nFold {TRAIN_FOLD}: train={len(train_df)}  val={len(val_df)}")
    print("  Train organ counts: " +
          "  ".join(f"{k}={v}" for k, v in sorted(organ_counts.items())))

    train_ds = HubMapDataset(train_df, is_train=True)
    val_ds   = HubMapDataset(val_df,   is_train=False)

    # Count-normalised WeightedRandomSampler.
    #
    # Problem: flat per-organ weights (lung=3, spleen=2) were designed for the
    # competition dataset (~70 samples/organ). Adding 534 Zenodo lung images
    # inflates lung's effective draws to ~1800 while spleen stays at ~140,
    # causing spleen Dice to drop from 0.847 → 0.652 in every Zenodo run.
    #
    # Fix: weight = MULTIPLIER / organ_count
    # Each organ's expected batch share = MULTIPLIER / sum_of_all(MULTIPLIER),
    # regardless of how many raw samples it has. Adding more lung samples no
    # longer dilutes spleen — the sampler automatically compensates.
    ORGAN_MULTIPLIERS = {
        'lung':           3.0,   # lung is sparse/noisy → oversample
        'spleen':         4.0,   # raised from 2.0 to fully counteract Zenodo flood
        'prostate':       2.0,
        'kidney':         1.5,
        'largeintestine': 1.0,
    }
    sample_weights = []
    for _, row in train_df.iterrows():
        organ      = row['organ']
        count      = organ_counts.get(organ, 1)
        multiplier = ORGAN_MULTIPLIERS.get(organ, 1.0)
        sample_weights.append(multiplier / count)

    w = torch.DoubleTensor(sample_weights)
    sampler = WeightedRandomSampler(w, len(w), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=4, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False,
                              num_workers=4, pin_memory=True)

    model     = build_model()
    if n_gpus > 1:
        print(f"  Wrapping in DataParallel across {n_gpus} GPUs")
        model = nn.DataParallel(model)
    model = model.to(DEVICE)

    criterion = FocalDiceLoss().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    total_steps = len(train_loader) // ACCUM * EPOCHS
    scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=1e-6)
    scaler    = torch.amp.GradScaler('cuda')

    best_val_loss = float('inf')
    best_dice     = 0.0
    no_improve    = 0
    ckpt_path     = MODELS_DIR / f'virchow_fold{TRAIN_FOLD}_best.pth'

    print(f"\nStarting training — {EPOCHS} epochs  "
          f"effective_batch={BATCH_SIZE * max(n_gpus,1) * ACCUM}\n")

    for epoch in range(EPOCHS):
        # ── Train ────────────────────────────────────────────────────────────
        model.train(); train_loss = 0.0; optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1:>3}/{EPOCHS} [train]",
                    dynamic_ncols=True)
        for step, batch in enumerate(pbar):
            imgs        = batch['img'].to(DEVICE)
            masks       = batch['mask'].to(DEVICE)
            organ_idx   = batch['organ_idx'].to(DEVICE)
            source_idx  = batch['source_idx'].to(DEVICE)
            pixel_scale = batch['pixel_scale'].to(DEVICE)
            with torch.amp.autocast('cuda'):
                pred = model(imgs, gt_organ=organ_idx,
                             gt_source=source_idx, gt_scale=pixel_scale)
                loss = criterion(pred, masks) / ACCUM
            scaler.scale(loss).backward()
            if (step+1) % ACCUM == 0 or (step+1) == len(train_loader):
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(); scheduler.step()
            train_loss += loss.item() * ACCUM
            pbar.set_postfix(loss=f"{train_loss/(step+1):.4f}",
                             lr=f"{optimizer.param_groups[0]['lr']:.2e}")

        # ── Validate ─────────────────────────────────────────────────────────
        model.eval(); val_loss = 0.0
        organ_dice = {o: [] for o in ORGAN_MAP}
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Ep {epoch+1:>3}/{EPOCHS} [val]  ",
                              dynamic_ncols=True):
                imgs        = batch['img'].to(DEVICE)
                masks       = batch['mask'].to(DEVICE)
                organ_idx   = batch['organ_idx'].to(DEVICE)
                source_idx  = batch['source_idx'].to(DEVICE)
                pixel_scale = batch['pixel_scale'].to(DEVICE)
                with torch.amp.autocast('cuda'):
                    pred      = model(imgs, gt_organ=organ_idx,
                                      gt_source=source_idx, gt_scale=pixel_scale)
                    val_loss += criterion(pred, masks).item()
                for i, organ in enumerate(batch['organ']):
                    organ_dice[organ].append(
                        dice_score(pred[i:i+1], masks[i:i+1]))

        avg_val  = val_loss / len(val_loader)
        per_organ = {o: float(np.mean(s)) for o, s in organ_dice.items() if s}
        avg_dice  = float(np.mean(list(per_organ.values())))
        organ_str = '  '.join(f"{k[:3]}={v:.3f}" for k, v in per_organ.items())

        print(f"\n  Ep {epoch+1:>3}  "
              f"train={train_loss/len(train_loader):.4f}  "
              f"val={avg_val:.4f}  dice={avg_dice:.4f}  "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")
        print(f"  {organ_str}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val; best_dice = avg_dice; no_improve = 0
            _m = model.module if isinstance(model, nn.DataParallel) else model
            torch.save({'epoch': epoch+1, 'state_dict': _m.state_dict(),
                        'val_loss': avg_val, 'dice': avg_dice,
                        'per_organ': per_organ}, ckpt_path)
            print(f"  ✓ NEW BEST  val={best_val_loss:.4f}  dice={best_dice:.4f}"
                  f"  → {ckpt_path}")
        else:
            no_improve += 1
            if no_improve >= 10:
                print(f"\n  Early stopping at epoch {epoch+1}")
                break

    print(f"\nBest: val_loss={best_val_loss:.4f}  mean_dice={best_dice:.4f}")
    return str(ckpt_path)

## Inference and Run

In [ ]:
def run_inference(ckpt_path):
    print(f"\nRunning inference (Virchow model)…")
    m = build_model().to(DEVICE)
    sd = torch.load(ckpt_path, map_location=DEVICE)
    m.load_state_dict(sd['state_dict'])
    m.eval()

    test_df  = pd.read_csv(DATA_DIR / 'test.csv')
    test_dir = DATA_DIR / 'test_images'

    THRESH = {'kidney': 0.40, 'prostate': 0.40, 'largeintestine': 0.35,
              'spleen': 0.40, 'lung': 0.30}

    results = []
    with torch.no_grad():
        for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
            img0 = _load_image_safe(str(test_dir / f"{row['id']}.tiff"),
                                    max_dim=max(IMG_SIZE)*2)
            if img0 is None:
                results.append({'id': row['id'], 'rle': ''}); continue
            img0 = ensure_3ch(img0)
            oh, ow = img0.shape[:2]
            acc = np.zeros((oh, ow), dtype='float32')
            n   = 0

            organ_idx   = torch.tensor(ORGAN_MAP.get(row['organ'], 0),
                                       dtype=torch.long).unsqueeze(0).to(DEVICE)
            source_idx  = torch.zeros(1, dtype=torch.long).to(DEVICE)
            pixel_scale = torch.tensor(
                ORGAN_PIXEL_SIZE_DEFAULTS.get(row['organ'], REF_PIXEL_SIZE) / REF_PIXEL_SIZE,
                dtype=torch.float32).unsqueeze(0).to(DEVICE)

            for flip in [False, True]:
                for k in range(4):
                    aug = img0.copy()
                    if flip: aug = aug[:, ::-1]
                    if k:    aug = np.rot90(aug, k)
                    inp = cv2.resize(aug, IMG_SIZE)
                    inp = torch.from_numpy(
                        preprocess_inputs(inp).transpose((2,0,1)).copy()
                    ).float().unsqueeze(0).to(DEVICE)
                    with torch.amp.autocast('cuda'):
                        logit = m(inp, gt_organ=organ_idx,
                                  gt_source=source_idx, gt_scale=pixel_scale)
                    p = torch.sigmoid(logit)[0,0].float().cpu().numpy()
                    if k:    p = np.rot90(p, 4-k)
                    if flip: p = p[:, ::-1]
                    acc += cv2.resize(p, (ow, oh)); n += 1

            acc /= n
            mask = (acc > THRESH.get(row['organ'], 0.4)).astype(np.uint8)
            results.append({'id': row['id'], 'rle': rle_encode(mask)})

    pd.DataFrame(results).to_csv(SUBMISSION_FILE, index=False)
    print(f"✓ Saved {SUBMISSION_FILE}  ({len(results)} rows)")

# ==============================================================================
# RUN
# ==============================================================================
ckpt = train()
run_inference(ckpt)
print(f"\nDone. Submit {SUBMISSION_FILE} to Kaggle.")